In [18]:
users = {
    1: {
        "name": "Alice",
        "email": "alice@example.com",
        "role": "user"
    },
    2: {
        "name": "Bob",
        "email": "bob@example.com",
        "role": "user"
    }
}

current_user_id = 1
requested_user_id = 2

current_user = users[current_user_id]

if requested_user_id != current_user_id and current_user["role"] != "admin":
    print("Access denied.")
else:
    print(users[requested_user_id])

Access denied.


In [19]:
accounts = {
    100: {
        "username": "Alice",
        "balance": 500
    },
    200: {
        "username": "Bob",
        "balance": 1000
    }
}

current_user = {
    "account_id": 100,
    "role": "user"
}

requested_account = 200

if (
    requested_account != current_user["account_id"]
    and current_user["role"] != "admin"
):
    print("Access denied.")
else:
    print(accounts[requested_account])

Access denied.


In [20]:
import hashlib
import secrets
import base64

def hash_password(password):
    salt = secrets.token_bytes(16)

    password_hash = hashlib.scrypt(
        password.encode("utf-8"),
        salt=salt,
        n=2**14,
        r=8,
        p=1
    )

    return (
        base64.b64encode(salt).decode()
        + ":"
        + base64.b64encode(password_hash).decode()
    )

password = "Password123"

stored_password = hash_password(password)

print("Secure password record:")
print(stored_password)

Secure password record:
89oKKtME3PW5mm/4/dYtCA==:jc6gcQwuwwefedJNWGzXFQac6VUj9jjZqUU0Q21J2FESTLa4dpAYfP/LE4d/7aeJhuDKzCFf0ZAnR1xVvUfrEw==


In [5]:
import hashlib
import secrets
import hmac

def create_password_record(password):
    salt = secrets.token_bytes(16)

    password_hash = hashlib.scrypt(
        password.encode("utf-8"),
        salt=salt,
        n=2**14,
        r=8,
        p=1
    )

    return salt, password_hash


def verify_password(password, salt, stored_hash):
    calculated_hash = hashlib.scrypt(
        password.encode("utf-8"),
        salt=salt,
        n=2**14,
        r=8,
        p=1
    )

    return hmac.compare_digest(
        calculated_hash,
        stored_hash
    )


salt, stored_hash = create_password_record("Password123")

print("Correct password:")
print(verify_password("Password123", salt, stored_hash))

print("Incorrect password:")
print(verify_password("WrongPassword", salt, stored_hash))

Correct password:
True
Incorrect password:
False


In [21]:
import sqlite3

connection = sqlite3.connect(":memory:")

connection.execute("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        username TEXT
    )
""")

connection.executemany(
    "INSERT INTO users (username) VALUES (?)",
    [
        ("Alice",),
        ("Bob",)
    ]
)

username = "' OR '1'='1"

cursor = connection.execute(
    "SELECT * FROM users WHERE username = ?",
    (username,)
)

results = cursor.fetchall()

print(results)

connection.close()

[]


In [22]:
def create_user_query(username):

    if not isinstance(username, str):
        raise ValueError("Username must be a string")

    if len(username) == 0:
        raise ValueError("Username cannot be empty")

    if len(username) > 50:
        raise ValueError("Username is too long")

    return {
        "username": username
    }


username = "Alice"

query = create_user_query(username)

print("Safe query:")
print(query)

Safe query:
{'username': 'Alice'}


In [23]:
import secrets
import time
import hashlib

users = {
    "alice@example.com": {
        "password": "OldPassword123"
    }
}

reset_tokens = {}


def create_reset_token(email):

    token = secrets.token_urlsafe(32)

    token_hash = hashlib.sha256(
        token.encode()
    ).hexdigest()

    reset_tokens[token_hash] = {
        "email": email,
        "expires": time.time() + 900
    }

    return token


def reset_password(email, token, new_password):

    token_hash = hashlib.sha256(
        token.encode()
    ).hexdigest()

    reset_request = reset_tokens.get(token_hash)

    if reset_request is None:
        return "Invalid reset token"

    if reset_request["email"] != email:
        return "Invalid reset token"

    if time.time() > reset_request["expires"]:
        del reset_tokens[token_hash]
        return "Reset token expired"

    if email not in users:
        return "Invalid reset request"

    users[email]["password"] = new_password

    del reset_tokens[token_hash]

    return "Password reset successfully"


token = create_reset_token(
    "alice@example.com"
)

print(
    reset_password(
        "alice@example.com",
        token,
        "NewSecurePassword"
    )
)

print(users)

Password reset successfully
{'alice@example.com': {'password': 'NewSecurePassword'}}


In [25]:
import hashlib


def verify_integrity(data, expected_hash):

    actual_hash = hashlib.sha256(data).hexdigest()

    return actual_hash == expected_hash


trusted_file = b"official library version"

expected_hash = hashlib.sha256(
    trusted_file
).hexdigest()


print(
    "Original file:",
    verify_integrity(
        trusted_file,
        expected_hash
    )
)

modified_file = b"modified malicious version"

print(
    "Modified file:",
    verify_integrity(
        modified_file,
        expected_hash
    )
)

Original file: True
Modified file: False


In [14]:
from urllib.parse import urlparse
import ipaddress


ALLOWED_HOSTS = {
    "example.com",
    "www.example.com"
}


def validate_url(url):

    parsed = urlparse(url)

    if parsed.scheme != "https":
        return False

    if parsed.hostname is None:
        return False

    hostname = parsed.hostname.lower()

    if hostname not in ALLOWED_HOSTS:
        return False

    try:
        address = ipaddress.ip_address(hostname)

        if (
            address.is_private
            or address.is_loopback
            or address.is_link_local
            or address.is_reserved
            or address.is_multicast
        ):
            return False

    except ValueError:
        pass

    return True


url = "https://example.com"

if validate_url(url):
    print("URL approved.")
else:
    print("URL rejected.")

URL approved.


In [26]:
import hashlib
import secrets
import hmac


def create_password_record(password):

    salt = secrets.token_bytes(16)

    password_hash = hashlib.scrypt(
        password.encode("utf-8"),
        salt=salt,
        n=2**14,
        r=8,
        p=1
    )

    return salt, password_hash


def verify_password(password, salt, stored_hash):

    calculated_hash = hashlib.scrypt(
        password.encode("utf-8"),
        salt=salt,
        n=2**14,
        r=8,
        p=1
    )

    return hmac.compare_digest(
        calculated_hash,
        stored_hash
    )


salt, stored_hash = create_password_record(
    "Password123"
)


input_password = "Password123"

if verify_password(
    input_password,
    salt,
    stored_hash
):
    print("Login success")
else:
    print("Login failed")

Login success
